In [4]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/vuelos_clima_2025q1.parquet")
df["estado"] = df["estado"].astype("category")
normal = df[df["estado"] == "normal"]

print("Forma:", df.shape)
print("\nKPIs globales (35 aeropuertos de origen, ene-mar 2025):")
print(f"  Vuelos totales:        {len(df):,}")
print(f"  % puntualidad (<15 min, vuelos normal): {(normal['ArrDel15'] == 0).mean():.1%}")
print(f"  Retraso medio llegada:   {normal['ArrDelay'].mean():.1f} min | mediana: {normal['ArrDelay'].median():.0f} min")
print(f"  % cancelados:            {(df['estado'] == 'cancelado').mean():.2%}")
print(f"  % desviados:             {(df['estado'] == 'desviado').mean():.2%}")

print("\nTipos conservados tras el parquet:")
print(df.dtypes.value_counts())

Forma: (1138858, 49)

KPIs globales (35 aeropuertos de origen, ene-mar 2025):
  Vuelos totales:        1,138,858
  % puntualidad (<15 min, vuelos normal): 79.8%
  Retraso medio llegada:   4.8 min | mediana: -7 min
  % cancelados:            1.74%
  % desviados:             0.25%

Tipos conservados tras el parquet:
float64           13
Float64            7
Int16              4
Int8               4
int64              3
bool               2
boolean            2
category           2
category           2
category           1
datetime64[ns]     1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
Name: count, dtype: int64


Las 1.138.858 filas y 49 columnas coinciden con el dataset final. El 79,8 % de los vuelos `normal` llega con menos de 15 min de retraso; la media (4,8 min) es mayor que la mediana (-7 min) por la cola de retrasos extremos ya descrita. Las tasas de cancelación (1,74 %) y desvío (0,25 %) también coinciden con el notebook 02. Los tipos se conservan correctamente, salvo `estado`, que quedó como `object` y se reconvierte a `category` a continuación

### 2. Retraso de llegada por aerolínea, franja horaria y día de la semana

Se compara la mediana de `ArrDelay` y el % de puntualidad (< 15 min) entre aerolíneas, franjas horarias y días de la semana, en los vuelos `normal`. Es un análisis descriptivo; la significación estadística se evaluará en el notebook 04.

In [5]:
dias = {1: "Lunes", 2: "Martes", 3: "Miércoles", 4: "Jueves", 5: "Viernes", 6: "Sábado", 7: "Domingo"}
normal = normal.assign(dia_semana=normal["DayOfWeek"].map(dias))

for col in ["Reporting_Airline", "franja_hora", "dia_semana"]:
    resumen = normal.groupby(col, observed=True).agg(
        vuelos=("ArrDelay", "size"),
        mediana_retraso=("ArrDelay", "median"),
        pct_puntual=("ArrDel15", lambda s: (s == 0).mean() * 100)
    ).round(1)
    print(f"--- {col} ---")
    print(resumen.sort_values("mediana_retraso", ascending=False), "\n")

--- Reporting_Airline ---
                   vuelos  mediana_retraso  pct_puntual
Reporting_Airline                                      
HA                   9969             -1.0         83.7
OH                  31803             -2.0         69.8
AS                  37827             -5.0         79.6
MQ                  35762             -6.0         80.3
AA                 179058             -7.0         78.6
B6                  39443             -7.0         74.7
OO                 107576             -7.0         78.2
WN                 212677             -7.0         82.2
F9                  37706             -7.0         74.9
DL                 181181             -8.0         80.1
UA                 145064             -9.0         82.2
NK                  43363             -9.0         81.1
G4                   4211             -9.0         80.0
YX                  50567            -10.0         82.0 

--- franja_hora ---
                  vuelos  mediana_retraso  pct_puntual
f

Por **aerolínea**, la puntualidad va del 69,8 % (OH) al 83,7 % (HA), un rango de 14 puntos, y la mediana de retraso oscila entre -1 y -10 min. No hay una diferencia extrema entre operadores.

Por **franja horaria**, la relación es más marcada y consistente: a medida que avanza el día empeora la puntualidad, de 91,2 % en madrugada a 74,8 % en noche. Es coherente con el efecto de acumulación de retrasos a lo largo de la jornada operativa.

Por **día de la semana**, martes y miércoles son los más puntuales (85,0 % y 83,4 %) y domingo el peor (75,1 %), aunque las diferencias son menores que por franja horaria.

Estas diferencias son descriptivas; su significación se contrastará en el notebook 04 con Kruskal-Wallis (más de dos grupos) y las comparaciones post-hoc correspondientes, dado que `ArrDelay` no sigue una distribución normal

## 3. Relación entre clima y retraso de llegada

Se compara la mediana de `ArrDelay` y el % de puntualidad según la intensidad de la precipitación, del viento y la presencia de clima adverso o nieve aproximada, en el aeropuerto de origen, para los vuelos `normal`.

In [6]:
for col in ["precip_cat", "viento_cat", "clima_adverso", "nieve_aprox"]:
    resumen = normal.groupby(col, observed=True).agg(
        vuelos=("ArrDelay", "size"),
        mediana_retraso=("ArrDelay", "median"),
        pct_puntual=("ArrDel15", lambda s: (s == 0).mean() * 100)
    ).round(1)
    print(f"--- {col} ---")
    print(resumen, "\n")

# Correlación de Spearman entre variables continuas de clima y retraso
clima_cols = ["prcp_orig", "wspd_orig", "tavg_orig"]
corr = normal[clima_cols + ["ArrDelay"]].corr(method="spearman")["ArrDelay"].drop("ArrDelay")
print("Correlación de Spearman con ArrDelay:")
print(corr.round(3))

--- precip_cat ---
            vuelos  mediana_retraso  pct_puntual
precip_cat                                      
Sin lluvia  837623             -8.0         83.1
Ligera      174981             -3.0         71.7
Moderada     63493             -1.0         68.3
Intensa      40110              0.0         65.9 

--- viento_cat ---
            vuelos  mediana_retraso  pct_puntual
viento_cat                                      
Flojo       652247             -8.0         82.2
Moderado    381810             -6.0         77.0
Fuerte       82150             -5.0         73.8 

--- clima_adverso ---
               vuelos  mediana_retraso  pct_puntual
clima_adverso                                      
False          957243             -8.0         81.8
True           158964             -1.0         67.9 

--- nieve_aprox ---
              vuelos  mediana_retraso  pct_puntual
nieve_aprox                                       
False        1072643             -8.0         80.8
True          

La relación es consistente y con gradiente claro en **precipitación**: la puntualidad baja de 83,1 % (sin lluvia) a 65,9 % (intensa), y la mediana pasa de -8 a 0 min. El **viento** muestra el mismo patrón, más suave: de 82,2 % (flojo) a 73,8 % (fuerte).

La variable **clima_adverso** separa bien los dos grupos: 81,8 % de puntualidad sin él frente a 67,9 % con él (14 puntos de diferencia). El efecto más marcado es el de **nieve_aprox**: con nieve la mediana de retraso es positiva (+9 min, frente a -8 min sin ella) y la puntualidad cae a 56,4 %, muy por debajo del resto de categorías.

Las correlaciones de Spearman confirman la dirección pero son débiles en magnitud: 0,137 para precipitación, 0,069 para viento y prácticamente nula para temperatura (-0,005), esperable porque el retraso depende también de la aerolínea, la franja horaria y la propagación de retrasos en la red aérea, no solo del clima puntual de origen.

Estas diferencias descriptivas se contrastarán en el notebook 04 con Mann-Whitney (grupos de dos, como `clima_adverso` y `nieve_aprox`) y Kruskal-Wallis (grupos de más de dos, como `precip_cat` y `viento_cat`), con el tamaño del efecto y la corrección por comparaciones múltiples correspondientes.